# 01 — Data Exploration

Khám phá cấu trúc dataset HuggingFace và kiểm tra chunks sau `run.py ingest`.

**Mục tiêu:**
- Xem cấu trúc dataset `phapdien-moj-gov-vn` và `anle-toaan-gov-vn`
- Kiểm tra chất lượng chunks
- Phân tích phân bố loại văn bản, SME score

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
from pathlib import Path
from collections import Counter

from vpl.settings import CHUNKS_FILE, LEGAL_DOCS_FILE, PRECEDENTS_FILE
print('Settings loaded')

## 1. Khám phá dataset HuggingFace

In [ ]:
from datasets import load_dataset

# Pháp điển
ds = load_dataset('tmquan/phapdien-moj-gov-vn', 'articles', split='train')
print('Pháp điển columns:', ds.column_names)
print('Num rows:', len(ds))
print('\nSample row:')
print(json.dumps(dict(ds[0]), ensure_ascii=False, indent=2)[:2000])

In [ ]:
# Án lệ
ds_anle = load_dataset('tmquan/anle-toaan-gov-vn', split='train')
print('Án lệ columns:', ds_anle.column_names)
print('Num rows:', len(ds_anle))
print('\nSample row:')
print(json.dumps(dict(ds_anle[0]), ensure_ascii=False, indent=2)[:2000])

## 2. Kiểm tra chunks sau ingest

In [ ]:
chunks = []
with CHUNKS_FILE.open('r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            chunks.append(json.loads(line))

print(f'Total chunks: {len(chunks)}')
print(f'\nSample chunk:')
print(json.dumps(chunks[0], ensure_ascii=False, indent=2))

In [ ]:
# Phân bố loại văn bản
doc_types = Counter(c['metadata']['doc_type'] for c in chunks)
print('Doc type distribution:')
for dtype, count in doc_types.most_common():
    print(f'  {dtype}: {count} ({count/len(chunks)*100:.1f}%)')

In [ ]:
# Submission eligible
eligible = [c for c in chunks if c['metadata']['submission_eligible']]
print(f'Eligible: {len(eligible)} / {len(chunks)} ({len(eligible)/len(chunks)*100:.1f}%)')
print(f'Unique docs: {len({c["metadata"]["formatted_doc"] for c in eligible})}')
print(f'Unique articles: {len({c["metadata"]["formatted_article"] for c in eligible})}')

In [ ]:
# SME score distribution
sme_scores = [c['metadata']['sme_score'] for c in chunks]
print(f'SME score: min={min(sme_scores):.1f}, max={max(sme_scores):.1f}, avg={sum(sme_scores)/len(sme_scores):.1f}')

print('\nTop 10 SME docs:')
for c in sorted(chunks, key=lambda x: x['metadata']['sme_score'], reverse=True)[:10]:
    m = c['metadata']
    print(f"  [{m['sme_score']:.1f}] {m['doc_id']} — {m['doc_title'][:60]}")

In [ ]:
# Kiểm tra format submission
import re
ARTICLE_FORMAT = re.compile(r'^[^|]+\|[^|]+\|Điều \d+[A-Za-z]?$')
invalid = [c for c in eligible if not ARTICLE_FORMAT.match(c['metadata']['formatted_article'])]
print(f'Invalid formatted_article: {len(invalid)}')
if invalid:
    for c in invalid[:3]:
        print(' ', c['metadata']['formatted_article'])